In [1]:
from pathlib import Path

MODEL_SCALE = "n"                 # "n" nebo "s"
IMAGE_SIZE = 640
DEVICE = 0                        # ID GPU (0 pro první GPU)
WORKERS = 4

# 1. Explicitní parametry pro srovnání s Kria DPU
CONF_OPERATING_POINT = 0.30       # Přesný práh pro provozní P/R/F1 na Krii
CONF_MAP_BASELINE = 0.001         # Nízký práh pro nezkreslenou mAP křivku
MATCHING_IOU_THRESHOLD = 0.50     # IoU práh pro párování predikcí s Ground Truth (PR křivka i CM = 0.50 jako na Krii)
NMS_IOU_THRESHOLD = 0.70          # IoU práh pro NMS (použije se pouze pokud není aktivní end2end one2one hlava)
MAX_DET = 300                     # Maximální počet detekcí na obrázek

# 4. Nastavení výkonnostních běhů (Throughput vs. Latency)
THROUGHPUT_BATCH_SIZE = 16        # Dávka pro měření maximální propustnosti
LATENCY_BATCH_SIZE = 1            # Dávka 1 pro měření latence (jako na Krii)
WARMUP_RUNS = 50                  # Počet zahřívacích iterací mimo měření
TIMED_RUNS = 500                  # Počet měřených obrázků

DATASET_DIR = Path("/kaggle/input/datasets/ondejpecina/openimages2classes/person_face_dataset")
CHECKPOINT_PATH = Path(f"/kaggle/input/models/ondejpecina/yolov26{MODEL_SCALE}2classes/pytorch/default/1")
WORK_DIR = Path("/kaggle/working")

In [2]:
%pip install -q ultralytics==8.4.150 nvidia-ml-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 596.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import hashlib
import json
import math
import time
import threading
import zipfile
import cv2
import numpy as np
import torch
import ultralytics
import yaml
from pathlib import Path
from ultralytics import YOLO

# Automatické dohledání datasetu
if not (DATASET_DIR / "dataset_manifest.json").is_file():
    found_manifests = list(Path("/kaggle/input").rglob("dataset_manifest.json"))
    if found_manifests:
        DATASET_DIR = found_manifests[0].parent
    else:
        raise FileNotFoundError(f"Dataset nebyl nalezen v: {DATASET_DIR}")

# Automatické dohledání nebo zabalení rozbaleného .pt checkpointu
def resolve_checkpoint(path: Path) -> Path:
    if path.is_file():
        return path
    if path.is_dir():
        pt_files = list(path.rglob("*.pt"))
        if pt_files:
            return pt_files[0]
        pkl_files = list(path.rglob("data.pkl"))
        if pkl_files:
            repacked_pt = WORK_DIR / f"repacked_yolo26{MODEL_SCALE}.pt"
            pkl_dir = pkl_files[0].parent
            print(f"Detekován rozbalený PyTorch checkpoint v {pkl_dir}. Balím do {repacked_pt}...")
            with zipfile.ZipFile(repacked_pt, "w", compression=zipfile.ZIP_STORED) as zf:
                for file in pkl_dir.rglob("*"):
                    if file.is_file():
                        zf.write(file, f"archive/{file.relative_to(pkl_dir)}")
            return repacked_pt
    all_pts = list(Path("/kaggle/input").rglob("*.pt"))
    if all_pts:
        return all_pts[0]
    raise FileNotFoundError(f"Nepodařilo se najít žádný .pt soubor ani rozbalený checkpoint v: {path}")

CHECKPOINT_PATH = resolve_checkpoint(CHECKPOINT_PATH)

# Výpočet SHA-256 hashe vah pro jednoznačnou identifikaci modelu
def compute_sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with filepath.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

CHECKPOINT_SHA256 = compute_sha256(CHECKPOINT_PATH)

manifest = json.loads((DATASET_DIR / "dataset_manifest.json").read_text())
CLASSES = manifest["classes"]
assert CLASSES == ["Person", "Human face"]

dataset_config = yaml.safe_load((DATASET_DIR / "person_face.yaml").read_text())
dataset_config["path"] = str(DATASET_DIR)
dataset_yaml = WORK_DIR / "person_face_runtime.yaml"
dataset_yaml.write_text(yaml.safe_dump(dataset_config, sort_keys=False))

print(f"Checkpoint: {CHECKPOINT_PATH} (SHA256: {CHECKPOINT_SHA256[:12]}...)")
print(f"Dataset: {DATASET_DIR} | Test obrázků: {manifest['counts']['test']}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Detekován rozbalený PyTorch checkpoint v /kaggle/input/models/ondejpecina/yolov26n2classes/pytorch/default/1/best. Balím do /kaggle/working/repacked_yolo26n.pt...
Checkpoint: /kaggle/working/repacked_yolo26n.pt (SHA256: 5d56b676c457...)
Dataset: /kaggle/input/datasets/ondejpecina/openimages2classes/person_face_dataset | Test obrázků: 5000


In [4]:
from ultralytics.models.yolo.detect import DetectionValidator

class KriaMatchingValidator(DetectionValidator):
    def init_metrics(self, model):
        super().init_metrics(model)
        orig_process_batch = self.confusion_matrix.process_batch
        target_conf = float(self.args.conf) if self.args.conf is not None else 0.25

        def process_batch_with_kria_iou(detections, batch, conf=target_conf, iou_thres=MATCHING_IOU_THRESHOLD):
            return orig_process_batch(detections, batch, conf=target_conf, iou_thres=MATCHING_IOU_THRESHOLD)

        self.confusion_matrix.process_batch = process_batch_with_kria_iou


class GPUPowerMonitor:
    def __init__(self, device_index=0, interval_sec=0.02):
        self.device_index = device_index
        self.interval = interval_sec
        self.samples = []
        self._stop_event = threading.Event()
        self._thread = None
        self.available = False
        try:
            import pynvml
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
            _ = pynvml.nvmlDeviceGetPowerUsage(self.handle)
            self.pynvml = pynvml
            self.available = True
        except Exception:
            self.available = False

    def _poll(self):
        while not self._stop_event.is_set():
            try:
                mw = self.pynvml.nvmlDeviceGetPowerUsage(self.handle)
                self.samples.append((time.perf_counter(), mw / 1000.0))
            except Exception:
                pass
            time.sleep(self.interval)

    def start(self):
        if self.available:
            self.samples.clear()
            self._stop_event.clear()
            self._thread = threading.Thread(target=self._poll, daemon=True)
            self._thread.start()

    def stop(self):
        if self.available and self._thread is not None:
            self._stop_event.set()
            self._thread.join()

    def summary(self, total_images, compute_wall_sec):
        if not self.available or not self.samples:
            return {
                "available": False,
                "scope": "unavailable",
                "measurement_window": "unavailable",
                "avg_power_watts": None,
                "energy_per_image_joules": None,
                "num_power_samples": 0,
            }
        watts = np.array([w for _, w in self.samples], dtype=np.float64)
        avg_w = float(np.mean(watts))

        trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
        if len(self.samples) >= 2 and trapz_fn is not None:
            times = np.array([t for t, _ in self.samples], dtype=np.float64)
            integrated_joules = float(trapz_fn(watts, times))
            sampled_duration_sec = float(times[-1] - times[0])
        else:
            integrated_joules = avg_w * compute_wall_sec
            sampled_duration_sec = compute_wall_sec

        total_joules = avg_w * compute_wall_sec
        return {
            "available": True,
            "scope": "GPU-only (NVML)",
            "measurement_window": "compute_only_excluding_disk_io (preprocess + inference + postprocess; images preloaded in RAM)",
            "compute_window_duration_sec": round(float(compute_wall_sec), 4),
            "nvml_sampled_span_sec": round(sampled_duration_sec, 4),
            "avg_power_watts": round(avg_w, 3),
            "min_power_watts": round(float(np.min(watts)), 3),
            "max_power_watts": round(float(np.max(watts)), 3),
            "total_energy_joules_avg_power_x_compute_time": round(total_joules, 4),
            "total_energy_joules_trapezoidal_integration": round(integrated_joules, 4),
            "energy_per_image_joules": round(total_joules / max(total_images, 1), 4),
            "energy_per_image_joules_trapezoidal": round(integrated_joules / max(total_images, 1), 4),
            "num_power_samples": len(self.samples),
        }


def inspect_execution_head_and_decoding(yolo_model, imgsz, max_det, device):
    """
    Ověří skutečný stav detekční hlavy před a po fúzi modelu v Ultralytics.
    U end2end=True (YOLO26) se při .fuse() odstraní one2many větev a zůstává čistě one2one Top-K bez NMS.
    """
    detect_head_unfused = yolo_model.model.model[-1]
    has_one2one_unfused = getattr(detect_head_unfused, "one2one_cv2", None) is not None
    has_one2many_unfused = getattr(detect_head_unfused, "cv2", None) is not None
    unfused_end2end_flag = bool(getattr(detect_head_unfused, "end2end", False))

    dummy = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    _ = yolo_model.predict(source=[dummy], imgsz=imgsz, max_det=max_det, device=device, half=False, verbose=False)

    fused_head = yolo_model.model.model[-1]
    end2end_active = bool(getattr(fused_head, "end2end", unfused_end2end_flag))
    predictor_nms_arg = getattr(yolo_model.predictor.args, "nms", None) if yolo_model.predictor else None

    # Pokud má hlava end2end=True, Ultralytics dekóduje přes Detect.postprocess (Top-K z one2one hlavy bez NMS)
    nms_enabled = (not end2end_active) or (predictor_nms_arg is True)

    return {
        "head_class": type(fused_head).__name__,
        "reg_max_dfl_bins": int(getattr(fused_head, "reg_max", 1)),
        "dfl_free": int(getattr(fused_head, "reg_max", 1)) == 1,
        "unfused_checkpoint_had_one2one_branch": bool(has_one2one_unfused),
        "unfused_checkpoint_had_one2many_branch": bool(has_one2many_unfused),
        "end2end_nms_free_head": bool(end2end_active and not nms_enabled),
        "nms_enabled": bool(nms_enabled),
        "decoding_strategy": "one2one_topk_nms_free" if (end2end_active and not nms_enabled) else "one2many_anchor_decode_plus_nms",
    }


def _compute_macro_summary(p_list, r_list, f1_list):
    if not p_list:
        return {"precision": None, "recall": None, "f1": None, "f1_harmonic_of_macro_pr": None, "f1_mean_of_per_class_f1": None}
    p_mac = float(np.mean(p_list))
    r_mac = float(np.mean(r_list))
    f1_harm = (2 * p_mac * r_mac / (p_mac + r_mac)) if (p_mac + r_mac) > 0 else 0.0
    f1_mean = float(np.mean(f1_list))
    return {
        "precision": round(p_mac, 6),
        "recall": round(r_mac, 6),
        "f1": round(f1_harm, 6),  # Sjednoceno s Kria evaluate.py (harmonický průměr macro P a macro R)
        "f1_harmonic_of_macro_pr": round(f1_harm, 6),
        "f1_mean_of_per_class_f1": round(f1_mean, 6),
    }


def extract_curve_interpolation_at_conf(val_metrics, target_conf: float, class_names: list):
    curves = getattr(val_metrics.box, "curves_results", None)
    per_class = {}
    p_all, r_all, f1_all = [], [], []

    if curves and len(curves) >= 4:
        _, f1_curve, p_curve, r_curve = curves[:4]
        x_conf = np.asarray(p_curve[0], dtype=np.float64)
        p_vals = np.asarray(p_curve[1], dtype=np.float64)
        r_vals = np.asarray(r_curve[1], dtype=np.float64)

        for idx, cname in enumerate(class_names):
            if idx < len(p_vals):
                pi = float(np.interp(target_conf, x_conf, p_vals[idx]))
                ri = float(np.interp(target_conf, x_conf, r_vals[idx]))
                f1i = (2 * pi * ri / (pi + ri)) if (pi + ri) > 0 else 0.0
                p_all.append(pi)
                r_all.append(ri)
                f1_all.append(f1i)
                per_class[cname] = {
                    "precision": round(pi, 6),
                    "recall": round(ri, 6),
                    "f1": round(f1i, 6),
                }

    return {
        "evaluated_confidence_threshold": target_conf,
        "matching_iou_threshold": 0.50,
        "macro_average": _compute_macro_summary(p_all, r_all, f1_all),
        "per_class": per_class,
    }


def extract_cm_metrics_at_conf(val_metrics, target_conf: float, matching_iou: float, class_names: list):
    cm_obj = getattr(val_metrics, "confusion_matrix", None)
    if cm_obj is None or getattr(cm_obj, "matrix", None) is None:
        return None

    mat = np.array(cm_obj.matrix, dtype=np.float64)
    per_class = {}
    precisions, recalls, f1s = [], [], []

    for idx, cname in enumerate(class_names):
        tp = float(mat[idx, idx])
        fp = float(mat[idx, :].sum() - tp)
        fn = float(mat[:, idx].sum() - tp)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
        per_class[cname] = {
            "precision": round(p, 6),
            "recall": round(r, 6),
            "f1": round(f1, 6),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
        }

    return {
        "evaluated_confidence_threshold": target_conf,
        "matching_iou_threshold": matching_iou,
        "macro_average": _compute_macro_summary(precisions, recalls, f1s),
        "per_class": per_class,
        "confusion_matrix": {
            "labels": class_names + ["background"],
            "rows_predicted_columns_true": cm_obj.matrix,
        },
    }


def benchmark_end_to_end(yolo_model, image_paths, batch_size, imgsz, conf, iou, max_det, device, warmup_runs, timed_runs):
    selected_paths = image_paths[:timed_runs]
    total_imgs = len(selected_paths)
    path_batches = [selected_paths[i:i + batch_size] for i in range(0, total_imgs, batch_size)]
    batch_counts = [len(b) for b in path_batches]

    # FÁZE A: Měření Disk I/O (mimo NVML power okno)
    preloaded_batches = []
    batch_io_ms = []
    for b_files in path_batches:
        t0_io = time.perf_counter()
        imgs = [cv2.imread(str(p)) for p in b_files]
        t1_io = time.perf_counter()
        batch_io_ms.append((t1_io - t0_io) * 1000.0)
        preloaded_batches.append(imgs)
    # Přesný součet I/O časů dávek (odstraňuje 0.0016 ms odchylku režie vnější smyčky)
    total_io_sum_sec = sum(batch_io_ms) / 1000.0

    # FÁZE B: Zahřátí GPU (Warm-up)
    dummy_img = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    dummy_batch = [dummy_img] * batch_size
    for _ in range(warmup_runs):
        _ = yolo_model.predict(
            source=dummy_batch, imgsz=imgsz, conf=conf, iou=iou, max_det=max_det,
            device=device, half=False, verbose=False
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # FÁZE C: Čisté výpočetní okno se synchronizací CUDA a NVML
    batch_e2e_no_io_ms = []
    batch_pre_ms = []
    batch_inf_ms = []
    batch_post_ms = []

    power_monitor = GPUPowerMonitor(device_index=device if isinstance(device, int) else 0)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    power_monitor.start()

    for imgs in preloaded_batches:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0_batch = time.perf_counter()

        preds = yolo_model.predict(
            source=imgs, imgsz=imgsz, conf=conf, iou=iou, max_det=max_det,
            device=device, half=False, verbose=False
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1_batch = time.perf_counter()

        compute_ms = (t1_batch - t0_batch) * 1000.0
        batch_e2e_no_io_ms.append(compute_ms)

        sp = preds[0].speed
        cur_bs = len(imgs)
        batch_pre_ms.append(sp.get("preprocess", 0.0) * cur_bs)
        batch_inf_ms.append(sp.get("inference", 0.0) * cur_bs)
        batch_post_ms.append(sp.get("postprocess", 0.0) * cur_bs)

    power_monitor.stop()

    batch_e2e_with_io_ms = [c + io for c, io in zip(batch_e2e_no_io_ms, batch_io_ms)]
    total_compute_sum_sec = sum(batch_e2e_no_io_ms) / 1000.0
    total_with_io_sum_sec = total_compute_sum_sec + total_io_sum_sec

    full_indices = [i for i, c in enumerate(batch_counts) if c == batch_size]
    partial_indices = [i for i, c in enumerate(batch_counts) if c != batch_size]

    def full_batch_stats(arr_ms):
        vals = np.asarray([arr_ms[i] for i in full_indices] if full_indices else arr_ms, dtype=np.float64)
        return {
            "mean_ms": round(float(np.mean(vals)), 4),
            "p50_ms": round(float(np.percentile(vals, 50)), 4),
            "p95_ms": round(float(np.percentile(vals, 95)), 4),
            "min_ms": round(float(np.min(vals)), 4),
            "max_ms": round(float(np.max(vals)), 4),
        }

    def amortized_image_stats(arr_ms):
        exact_mean_ms = float(np.sum(arr_ms)) / total_imgs
        per_batch_per_img = np.asarray([t / c for t, c in zip(arr_ms, batch_counts)], dtype=np.float64)
        per_image_expanded = np.repeat(per_batch_per_img, batch_counts)
        return {
            "mean_ms": round(exact_mean_ms, 4),
            "p50_ms": round(float(np.percentile(per_image_expanded, 50)), 4),
            "p95_ms": round(float(np.percentile(per_image_expanded, 95)), 4),
            "min_ms": round(float(np.min(per_image_expanded)), 4),
            "max_ms": round(float(np.max(per_image_expanded)), 4),
        }

    partial_info = None
    if partial_indices:
        idx = partial_indices[0]
        partial_info = {
            "batch_index": idx,
            "images_in_partial_batch": batch_counts[idx],
            "end_to_end_excluding_io_ms": round(batch_e2e_no_io_ms[idx], 4),
            "per_image_excluding_io_ms": round(batch_e2e_no_io_ms[idx] / batch_counts[idx], 4),
        }

    return {
        "configured_batch_size": batch_size,
        "warmup_iterations": warmup_runs,
        "total_batches_executed": len(path_batches),
        "full_batches_count": len(full_indices),
        "partial_final_batch": partial_info,
        "actual_processed_images": total_imgs,
        "image_loading_included_in_primary_e2e": False,
        "image_saving_included": False,
        "total_wall_time_sec_excluding_disk_io": round(total_compute_sum_sec, 4),
        "total_disk_io_time_sec": round(total_io_sum_sec, 4),
        "total_wall_time_sec_with_disk_io": round(total_with_io_sum_sec, 4),
        "measured_fps_excluding_disk_io": round(total_imgs / total_compute_sum_sec, 3),
        "measured_fps_with_disk_io": round(total_imgs / total_with_io_sum_sec, 3),
        "full_batch_latency_ms": {
            "note": f"Computed strictly over {len(full_indices)} full batches of size {batch_size}.",
            "preprocess": full_batch_stats(batch_pre_ms),
            "inference": full_batch_stats(batch_inf_ms),
            "postprocess": full_batch_stats(batch_post_ms),
            "end_to_end_excluding_io": full_batch_stats(batch_e2e_no_io_ms),
            "disk_image_load_io": full_batch_stats(batch_io_ms),
            "end_to_end_with_io": full_batch_stats(batch_e2e_with_io_ms),
        },
        "amortized_per_image_ms": {
            "note": f"mean_ms is total_time_ms / {total_imgs} actual processed images (consistent with measured_fps).",
            "preprocess": amortized_image_stats(batch_pre_ms),
            "inference": amortized_image_stats(batch_inf_ms),
            "postprocess": amortized_image_stats(batch_post_ms),
            "end_to_end_excluding_io": amortized_image_stats(batch_e2e_no_io_ms),
            "disk_image_load_io": amortized_image_stats(batch_io_ms),
            "end_to_end_with_io": amortized_image_stats(batch_e2e_with_io_ms),
        },
        "power_and_energy": power_monitor.summary(total_imgs, total_compute_sum_sec),
    }

In [5]:
fp32_model = YOLO(str(CHECKPOINT_PATH))

# Ověření skutečné detekční hlavy (one2one vs one2many) a dekódovací strategie po fúzi
head_verification = inspect_execution_head_and_decoding(
    YOLO(str(CHECKPOINT_PATH)), imgsz=IMAGE_SIZE, max_det=MAX_DET, device=DEVICE
)
print("Ověření hlavy a dekódování:", head_verification)

# --- BĚH 1: Standardní mAP vyhodnocení (nízký práh conf=0.001, nezkreslená PR křivka, batch=16) ---
print(f"\n[1/4] Spouštím standardní mAP evaluaci (conf={CONF_MAP_BASELINE}, batch={THROUGHPUT_BATCH_SIZE})...")
t0_val_map = time.perf_counter()
map_metrics = fp32_model.val(
    validator=KriaMatchingValidator,
    data=str(dataset_yaml),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=THROUGHPUT_BATCH_SIZE,
    conf=CONF_MAP_BASELINE,
    iou=NMS_IOU_THRESHOLD,
    max_det=MAX_DET,
    half=False,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(WORK_DIR / "runs"),
    name=f"yolo26{MODEL_SCALE}_fp32_test_map_conf0001",
)
val_map_wall_sec = time.perf_counter() - t0_val_map

# --- BĚH 2: Evaluace na pevném provozním prahu conf=0.30 a matching IoU=0.50 ---
print(f"\n[2/4] Spouštím evaluaci pro provozní bod (conf={CONF_OPERATING_POINT}, matching IoU={MATCHING_IOU_THRESHOLD})...")
t0_val_op = time.perf_counter()
op_metrics = fp32_model.val(
    validator=KriaMatchingValidator,
    data=str(dataset_yaml),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=THROUGHPUT_BATCH_SIZE,
    conf=CONF_OPERATING_POINT,
    iou=NMS_IOU_THRESHOLD,
    max_det=MAX_DET,
    half=False,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(WORK_DIR / "runs"),
    name=f"yolo26{MODEL_SCALE}_fp32_test_op_conf030",
)
val_op_wall_sec = time.perf_counter() - t0_val_op

# Extrakce P/R/F1 na prahu 0.30:
# 1) Exaktně z Confusion Matrix spočtené na prahu conf=0.30 a matching IoU=0.50
cm_at_conf_030 = extract_cm_metrics_at_conf(op_metrics, CONF_OPERATING_POINT, MATCHING_IOU_THRESHOLD, CLASSES)
# 2) Čistou interpolací z PR křivky (bez míchání CM hodnot z běhu 0.001)
curve_at_conf_030_untruncated = extract_curve_interpolation_at_conf(map_metrics, CONF_OPERATING_POINT, CLASSES)
curve_at_conf_030_op_run = extract_curve_interpolation_at_conf(op_metrics, CONF_OPERATING_POINT, CLASSES)

# --- BĚH 3 & 4: Měření End-to-End FPS, latence (p50/p95) a spotřeby pro Batch 16 a Batch 1 ---
test_img_dir = DATASET_DIR / "images" / "test"
test_image_paths = sorted([p for p in test_img_dir.iterdir() if p.is_file()])

print(f"\n[3/4] Měření výkonu a latence pro Batch={THROUGHPUT_BATCH_SIZE} (Throughput)...")
perf_batch16 = benchmark_end_to_end(
    fp32_model, test_image_paths,
    batch_size=THROUGHPUT_BATCH_SIZE, imgsz=IMAGE_SIZE,
    conf=CONF_OPERATING_POINT, iou=NMS_IOU_THRESHOLD, max_det=MAX_DET,
    device=DEVICE, warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS
)

print(f"\n[4/4] Měření výkonu a latence pro Batch={LATENCY_BATCH_SIZE} (Single-Image Latency)...")
perf_batch1 = benchmark_end_to_end(
    fp32_model, test_image_paths,
    batch_size=LATENCY_BATCH_SIZE, imgsz=IMAGE_SIZE,
    conf=CONF_OPERATING_POINT, iou=NMS_IOU_THRESHOLD, max_det=MAX_DET,
    device=DEVICE, warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS
)

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_ready(v) for v in value]
    if hasattr(value, "tolist"):
        return json_ready(value.tolist())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

box_map = map_metrics.box
per_class_ap = {}
for row_index, class_index in enumerate(box_map.ap_class_index):
    class_index = int(class_index)
    per_class_ap[CLASSES[class_index]] = {
        f"AP@{0.50 + 0.05 * i:.2f}": float(ap)
        for i, ap in enumerate(box_map.all_ap[row_index])
    }

gpu_name = torch.cuda.get_device_name(DEVICE) if torch.cuda.is_available() else "CPU"

report = {
    "model_scale": MODEL_SCALE,
    "checkpoint": {
        "path": str(CHECKPOINT_PATH),
        "sha256": CHECKPOINT_SHA256,
    },
    "execution_metadata": {
        "gpu_model": gpu_name,
        "num_gpus_visible": torch.cuda.device_count(),
        "device_used": DEVICE,
        "inference_dtype": "float32",
        "amp_enabled": False,
        "half_precision": False,
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "ultralytics_version": ultralytics.__version__,
        "warmup_runs": WARMUP_RUNS,
        "timed_runs_images": min(TIMED_RUNS, len(test_image_paths)),
    },
    "explicit_evaluation_settings": {
        "operating_point_confidence": CONF_OPERATING_POINT,
        "map_baseline_confidence": CONF_MAP_BASELINE,
        "evaluation_matching_iou_threshold": MATCHING_IOU_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "nms_iou_active_in_execution": head_verification["nms_enabled"],
        "max_det": MAX_DET,
        "nms_enabled": head_verification["nms_enabled"],
        "end2end_nms_free_head": head_verification["end2end_nms_free_head"],
        "decoding_strategy": head_verification["decoding_strategy"],
        "head_verification_details": head_verification,
        "imgsz": IMAGE_SIZE,
        "throughput_batch_size": THROUGHPUT_BATCH_SIZE,
        "latency_batch_size": LATENCY_BATCH_SIZE,
    },
    "dataset": {
        "name": "Open Images V7 test split",
        "class_names": CLASSES,
        "num_images": manifest["counts"]["test"],
        "test_image_filenames": manifest["test_image_filenames"],
    },
    "accuracy_standard_map_untruncated_conf_0001": {
        "description": "Standard mAP evaluation with conf=0.001 so the Precision-Recall curve is not truncated.",
        "overall": map_metrics.results_dict,
        "per_class_summary_at_best_f1": map_metrics.summary(),
        "per_class_ap_by_iou": per_class_ap,
        "validation_wall_time_sec": round(val_map_wall_sec, 3),
        "validator_speed_ms_per_image": getattr(map_metrics, "speed", None),
        "run_directory": str(map_metrics.save_dir),
    },
    "accuracy_operating_point_conf_030": {
        "description": (
            f"Precision, Recall, F1 and Confusion Matrix evaluated strictly at confidence={CONF_OPERATING_POINT} "
            f"and matching IoU={MATCHING_IOU_THRESHOLD} (matching Kria evaluation criteria)."
        ),
        "from_confusion_matrix_at_conf_030_iou_050": cm_at_conf_030,
        "from_pr_curve_interpolation_at_conf_030_iou_050": {
            "from_untruncated_0001_pr_curve": curve_at_conf_030_untruncated,
            "from_conf_030_run_pr_curve": curve_at_conf_030_op_run,
        },
        "ultralytics_best_f1_summary_above_030_floor": {
            "note": "Ultralytics results_dict selects best-F1 operating point >= 0.30; kept for reference only.",
            "results_dict": op_metrics.results_dict,
        },
        "validation_wall_time_sec": round(val_op_wall_sec, 3),
        "run_directory": str(op_metrics.save_dir),
    },
    "performance_benchmarks": {
        "batch_16_throughput_run": perf_batch16,
        "batch_1_latency_run": perf_batch1,
    },
}

report_path = WORK_DIR / f"yolo26{MODEL_SCALE}_fp32_kria_comparison_metrics.json"
report_path.write_text(json.dumps(json_ready(report), indent=2, ensure_ascii=False) + "\n")

print("\n==========================================")
print("Hotovo! Opravený JSON report uložen do:", report_path)
print("------------------------------------------")
print(f"Strategie dekódování:  {head_verification['decoding_strategy']} (end2end={head_verification['end2end_nms_free_head']}, NMS={head_verification['nms_enabled']})")
print(f"mAP50 (conf=0.001):    {map_metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95 (conf=0.001): {map_metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"P/R/F1 @ conf=0.30, IoU=0.50 (CM):     {cm_at_conf_030['macro_average']['precision']:.4f} / "
      f"{cm_at_conf_030['macro_average']['recall']:.4f} / {cm_at_conf_030['macro_average']['f1']:.4f}")
print(f"P/R/F1 @ conf=0.30, IoU=0.50 (Křivka): {curve_at_conf_030_untruncated['macro_average']['precision']:.4f} / "
      f"{curve_at_conf_030_untruncated['macro_average']['recall']:.4f} / {curve_at_conf_030_untruncated['macro_average']['f1']:.4f}")
print(f"Batch 16 FPS (bez I/O): {perf_batch16['measured_fps_excluding_disk_io']} FPS "
      f"(amortizovaný průměr: {perf_batch16['amortized_per_image_ms']['end_to_end_excluding_io']['mean_ms']} ms/obr.)")
print(f"Batch 1  FPS (bez I/O): {perf_batch1['measured_fps_excluding_disk_io']} FPS "
      f"(průměr: {perf_batch1['amortized_per_image_ms']['end_to_end_excluding_io']['mean_ms']} ms/obr.)")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ověření hlavy a dekódování: {'head_class': 'Detect', 'reg_max_dfl_bins': 1, 'dfl_free': True, 'unfused_checkpoint_had_one2one_branch': True, 'unfused_checkpoint_had_one2many_branch': True, 'end2end_nms_free_head': False, 'nms_enabled': True, 'decoding_strategy': 'one2many_anchor_decode_plus_nms'}

[1/4] Spouštím standardní mAP evaluaci (conf=0.001, batch=16)...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n_dpu summary (fused): 210 layers, 2,388,346 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 0.3±0.2 MB/s, size: 321.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/ondejpecina